In [ ]:
"""
RoleBasedRAG/
│
├── app/
│   ├── main.py                     # FastAPI entry
│   │
│   ├── schemas/
│   │   ├── request.py              # query models
│   │   ├── response.py
│   │
│   ├── services/
│   │   ├── rag_service.py         # RAG pipeline (CORE)
│   │   ├── ingestion_service.py   # load + chunk + embed
│   │   ├── retrieval_service.py   # Qdrant search
│   │   ├── llm_service.py         # Groq wrapper
│   │
│   ├── utils/
│   │   ├── rbac.py                # role filtering logic
│   │   ├── guardrails.py          # PII + injection detection
│   │   ├── embeddings.py          # embedding model
│   │   ├── logger.py              # cost + monitoring
|   |   ├── vectorstore.py
│
├── resources/data/
│   ├── engineering/
│   ├── finance/
│   ├── hr/
│   ├── marketing/
│   ├── general/
│
├── vector_db/ (NEW - optional local Qdrant config)
├── eval/
│   ├── ragas_eval.py              # evaluation pipeline
│
├── frontend/ (React app)
│
├── docker/
├── requirements.txt
"""

In [ ]:
"""🧠 5. How your system should work (REAL FLOW)

This is what your project SHOULD do:

User Query
   ↓
FastAPI (main.py)
   ↓
guardrails.py (check safety)
   ↓
rbac.py (get allowed departments)
   ↓
retrieval_service.py (Qdrant filtered search)
   ↓
rag_service.py
   ↓
llm_service.py (Groq)
   ↓
Response
   ↓
logger.py (LangSmith + cost tracking)
"""

In [ ]:
"""
🧠 6. What each file should do (VERY IMPORTANT)
    🔹 rag_service.py (CORE ORCHESTRATOR)
        calls everything in correct order
        THIS is your “brain”

    🔹 retrieval_service.py
        connects to Qdrant
        applies RBAC filters
        returns top-k chunks

    🔹 ingestion_service.py
        reads:
            resources/data/finance/*
            resources/data/hr/*
        chunks text
        creates embeddings
        pushes to Qdrant with metadata:
            {
            "text": "...",
            "department": "finance"
            }
    🔹 rbac.py
        Very important:
            ROLE_MAP = {
                "hr": ["hr", "general"],
                "finance": ["finance", "general"],
                "marketing": ["marketing", "general"],
                "exec": ["hr", "finance", "marketing", "engineering", "general"]
            }

    🔹 guardrails.py
        Handles:
            PII detection
            prompt injection detection

    🔹 llm_service.py
        Wraps:
            langchain_groq.ChatGroq
⚙️ 7. WHAT YOU SHOULD BUILD NEXT (ORDER)
    Follow this EXACT order:
    ✅ STEP 1 — Add Qdrant
        Install Docker Qdrant or cloud
        Create collection
    ✅ STEP 2 — Build ingestion pipeline
        read resources/data
        chunk + embed
        store with department tag
    ✅ STEP 3 — Build RBAC filtering
        filter by metadata in Qdrant
    ✅ STEP 4 — Build RAG service
        retrieval + prompt + LLM
    ✅ STEP 5 — FastAPI endpoints
        POST /chat
        POST /ingest
    ✅ STEP 6 — React frontend
        role selector
        chat UI
    ✅ STEP 7 — Add observability
        LangSmith tracing
        logs
    ✅ STEP 8 — Add RAGAS evaluation
        run after each deployment

🚀 8. IMPORTANT INSIGHT (Interview GOLD)
    If asked in interview:
        “Why did you structure it this way?”
    You say:
        “I separated ingestion, retrieval, RBAC, and LLM layers so each component can scale independently. This also allows swapping vector DB or LLM without affecting business logic.”
"""

In [ ]:
#Vectorstore
"""
->os: used to read environment variables
->QdrantClient: Python client for the Qdrant vector database
->Distance, VectorParams: configuration for how vectors are stored and compared
->QdrantVectorStore: LangChain wrapper to use Qdrant easily
->embeddings: your embedding model (likely something like OpenAI / HuggingFace embeddings)

2. Connect to Qdrant server
    client = QdrantClient(url=os.getenv("QDRANT_URL"))
    Connects to a Qdrant instance running somewhere (cloud or local)

3. Get collection name
    collection_name = os.getenv("QDRANT_COLLECTION_NAME")
    A collection in Qdrant is like a table in a database
    This stores your vectors (embeddings + metadata)

4. Create collection if it doesn’t exist
        if not client.collection_exists(collection_name):
    Checks if the vector collection already exists.
    If not, it creates one:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(
                size=int(os.getenv("EMBEDDING_DIMENSIONS", 384)),
                distance=Distance.COSINE
            ),
        )
    Key parts:
    size: dimension of each embedding vector
        Example: 384, 768, 1536 depending on model
        Must match your embedding model output exactly
    Distance.COSINE:
        Means similarity is measured using cosine similarity
        Common for text embeddings

    So this creates a vector index like:
        “Store vectors of length 384 and compare them using cosine similarity.”
5. Create LangChain vector store wrapper
    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embeddings
    )
    This is the bridge between LangChain and Qdrant.
    It allows you to:
    Add documents:
        vectorstore.add_texts([...])
    Search similar text:
        vectorstore.similarity_search("query")
"""

In [ ]:
#DATA INGESTION
"""
🧠 Imagine this setup
You have a folder:
    data/hr_docs/
Inside it:
    leave_policy.pdf
    team_rules.md
    employees.csv
And you call:
    ingest_folder("data/hr_docs", department="hr")

    🔷 STEP 1: Folder starts processing
        📂 Processing folder: data/hr_docs
    The function now loops through each file.

    🔷 STEP 2: File 1 → PDF (leave_policy.pdf)
        📄 Extract text
        Imagine PDF contains:
            Employees get 20 days leave per year.
            Sick leave is separate.
        ✂️ Chunking happens:
        Chunk 1:
            "Employees get 20 days leave per year. Sick leave is separate."
        📦 Converted to Document:
            Document(
                page_content="Employees get 20 days leave per year. Sick leave is separate.",
                metadata={
                    "department": "hr",
                    "source": "leave_policy.pdf"
                }
            )

    🔷 STEP 3: File 2 → Markdown (team_rules.md)
        Content:
            Team must report by 10 AM.
            Work from home allowed twice a week.
        Chunking:
            Chunk 1:
                "Team must report by 10 AM. Work from home allowed twice a week."
        Document created:
            Document(
                page_content="Team must report by 10 AM. Work from home allowed twice a week.",
                metadata={
                    "department": "hr",
                    "source": "team_rules.md"
                }
            )

    🔷 STEP 4: File 3 → CSV (employees.csv)
        CSV content:
            | name | role     |
            | ---- | -------- |
            | Ravi | Engineer |
            | Anil | HR       |
        Each row becomes a document:
        Row 0:
            name: Ravi | role: Engineer
        Row 1:
            name: Anil | role: HR
        Documents created:
            Document(
                page_content="name: Ravi | role: Engineer",
                metadata={
                    "department": "hr",
                    "source": "employees.csv",
                    "row": 0
                }
            )
            Document(
                page_content="name: Anil | role: HR",
                metadata={
                    "department": "hr",
                    "source": "employees.csv",
                    "row": 1
                }
            )

    🔷 STEP 5: Final collection before storing
        Now everything is collected:
            [
            leave_policy chunk,
            team_rules chunk,
            employee row 1,
            employee row 2
            ]
        Total docs:
            📦 TOTAL DOCS READY: 4

    🔷 STEP 6: Stored in vector DB
            vectorstore.add_documents(all_docs)
        Now each document becomes:
        Step inside Qdrant:
            text → embedding vector → stored with metadata
            
    🔷 FINAL RESULT IN DATABASE
        | Vector | Text              | Department | Source |
        | ------ | ----------------- | ---------- | ------ |
        | v1     | leave policy text | hr         | pdf    |
        | v2     | team rules        | hr         | md     |
        | v3     | Ravi Engineer     | hr         | csv    |
        | v4     | Anil HR           | hr         | csv    |

    🔷 STEP 7: Later search example 🔍
        User asks:
            “What is leave policy?”
        System does:
        1. Convert query → vector
            "leave policy" → embedding
        2. Search Qdrant
        If filtered:
            filter={"department": "hr"}
        It only searches HR data:
            ✔ leave_policy.pdf
            ✔ team_rules.md
            ✔ employee.csv

    🧠 Simple Flow Summary
        Folder
        ↓
        Read files (PDF / MD / CSV)
        ↓
        Extract text
        ↓
        Split into chunks (or rows for CSV)
        ↓
        Attach metadata (department = "hr")
        ↓
        Convert to Documents
        ↓
        Send to vector DB
        ↓
        Embeddings stored
    🔥 One-line explanation
        This method takes files from a folder, converts them into small searchable text pieces, tags them with a department, and stores them in a vector database so you can later search them using natural language.
"""